# Lab L2-2：U-Net 告訴我瑕疵在哪、有多大（D2-2）

**需求單 #D2-2**（虛構公司：曜嶼電子，製造品保部維修站）：不只要知道「這片有問題」，還要每個瑕疵的**位置與面積**。
影像用公開資料集 **VisA 的 PCB1**（CC BY 4.0，Amazon spot-diff 倉庫）。

| 步驟 | 做什麼 | 輸出 |
|---|---|---|
| 1 | 讀資料：良品用來訓練；不良品＋**真實遮罩**只用來測試 | 張數 |
| 2 | 合成瑕疵（DRAEM 想法）：在良品上貼程式產生的紋理 | 合成瑕疵條帶圖 |
| 3 | U-Net（三層）：每一塊的 shape | shape 表 |
| 4 | 訓練：BCE ＋ Dice | loss 曲線 |
| 5 | 在**真實遮罩**上量 IoU、Dice；面積換算 mm² | 疊圖、IoU／Dice |
| 6 | Gradio：上傳 → 遮罩疊圖＋面積 | 網頁 |

> 狀態：本筆記本尚未在 Colab T4 實跑（見 `research/lab-runs.md`）。

In [ ]:
# 第 1 格：環境
%pip install -q -U gradio
import torch, os, glob, random, platform, time
import numpy as np
print("torch", torch.__version__, "| python", platform.python_version())
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0), f"| 可用 VRAM {free/1e9:.1f} / {total/1e9:.1f} GB")
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SIZE = 256          # 沒有 GPU：改成 64
EPOCHS = 20         # 沒有 GPU：改成 2

## 1. 資料：VisA PCB1（和 D2-1 同一份）
解壓後的結構：`pcb1/Data/Images/Normal`、`pcb1/Data/Images/Anomaly`、`pcb1/Data/Masks/Anomaly`（遮罩和不良品同檔名，副檔名 .png）。
已經有 D2-1 的資料夾就不用再下載；講師若放在雲端硬碟，改 `PCB1_DIR`。

In [ ]:
VISA_URL = "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"  # 約 1.9 GB（2026-11-24 確認）
PCB1_DIR = "visa/pcb1"

if not os.path.isdir(PCB1_DIR):
    os.makedirs("visa", exist_ok=True)
    if not os.path.exists("visa/VisA.tar"):
        os.system(f"wget -q -O visa/VisA.tar {VISA_URL}")
    os.system("tar -xf visa/VisA.tar -C visa --wildcards 'pcb1/*'")
normal = sorted(glob.glob(f"{PCB1_DIR}/Data/Images/Normal/*.*"))
anomaly = sorted(glob.glob(f"{PCB1_DIR}/Data/Images/Anomaly/*.*"))
def mask_path(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    return f"{PCB1_DIR}/Data/Masks/Anomaly/{stem}.png"
anomaly = [p for p in anomaly if os.path.exists(mask_path(p))]
print("良品", len(normal), "張；有遮罩的不良品", len(anomaly), "張")

**切分**：訓練只用良品（瑕疵是我們自己合成的）；真實不良品與遮罩**只用來測試**，模型訓練時從沒看過。

In [ ]:
from PIL import Image
from torchvision import transforms

random.shuffle(normal)
n_train = int(len(normal) * 0.85)
tf = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])
tf_mask = transforms.Compose([transforms.Resize((SIZE, SIZE), interpolation=transforms.InterpolationMode.NEAREST),
                              transforms.ToTensor()])

t0 = time.time()
X_good = torch.stack([tf(Image.open(p).convert("RGB")) for p in normal[:n_train]])
X_bad = torch.stack([tf(Image.open(p).convert("RGB")) for p in anomaly])
M_bad = torch.stack([(tf_mask(Image.open(mask_path(p)).convert("L")) > 0).float() for p in anomaly])  # 多類別遮罩 → 0／1
print(f"讀取完成 {time.time()-t0:.0f}s | 訓練良品 {tuple(X_good.shape)} | 測試不良品 {tuple(X_bad.shape)} | 遮罩 {tuple(M_bad.shape)}")
print(f"真實瑕疵平均只佔畫面的 {M_bad.mean().item():.2%}")

## 2. 合成瑕疵（DRAEM 的想法）
沒有瑕疵遮罩，就自己做：
1. **遮罩**：低解析度隨機雜訊 → 放大變平滑 → 取最亮的一小塊，變成不規則的斑塊
2. **紋理**：程式產生的彩色雜訊（不用來路不明的紋理資料集）
3. **混合**：斑塊內換成「原圖和紋理的混合」，斑塊外不動

這樣每一張合成瑕疵都**自帶正確答案**（遮罩）。

In [ ]:
import torch.nn.functional as F

def blob_mask(size, max_frac=0.04):
    """平滑雜訊取最亮的一塊 → 不規則斑塊，約佔畫面 0.5%–max_frac。"""
    low = torch.rand(1, 1, random.choice([4, 8, 16]), random.choice([4, 8, 16]))
    smooth = F.interpolate(low, size=(size, size), mode="bicubic", align_corners=False)[0, 0]
    frac = random.uniform(0.005, max_frac)
    thr = smooth.flatten().kthvalue(int((1 - frac) * size * size)).values
    return (smooth > thr).float()                                   # [H, W]

def texture(size):
    """程式產生的紋理：幾種顏色的平滑雜訊。"""
    low = torch.rand(1, 3, random.choice([8, 16, 32]), random.choice([8, 16, 32]))
    return F.interpolate(low, size=(size, size), mode="bilinear", align_corners=False)[0]   # [3, H, W]

def synth_defect(x):
    """x: [3, H, W] 良品 → (合成瑕疵影像, 遮罩 [1, H, W])。一半的機率不加瑕疵（教模型也要說「沒有」）。"""
    if random.random() < 0.5:
        return x, torch.zeros(1, *x.shape[1:])
    # TODO：m = blob_mask(H)；t = texture(H)；beta 在 0.1–0.6 之間；斑塊內換成 beta*x + (1-beta)*t，斑塊外不動
    ...
    return x_aug.clamp(0, 1), m.unsqueeze(0)

import matplotlib.pyplot as plt
random.seed(1)
m = blob_mask(SIZE); t = texture(SIZE)
x0 = X_good[0]; xs = (x0 * (1 - m) + (0.3 * x0 + 0.7 * t) * m).clamp(0, 1)
fig, ax = plt.subplots(1, 4, figsize=(12, 3.2))
for a, img, title in zip(ax, [x0.permute(1, 2, 0), m, t.permute(1, 2, 0), xs.permute(1, 2, 0)],
                         ["good", "mask", "texture", "synthetic defect"]):
    a.imshow(img, cmap="gray" if img.ndim == 2 else None); a.set_title(title); a.axis("off")
plt.tight_layout(); plt.savefig("synth_strip.png", dpi=120); plt.show()

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SynthDefects(Dataset):
    """每次取一張良品，當場合成瑕疵 → 每個 epoch 的瑕疵都不一樣。"""
    def __init__(self, X):
        self.X = X
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x = self.X[i]
        if random.random() < 0.5:
            x = x.flip(-2, -1)                   # 旋轉 180°：VisA PCB1 良品約八成倒著拍，兩種方向都要學（D2-1 的資料陷阱）
        return synth_defect(x)

train_loader = DataLoader(SynthDefects(X_good), batch_size=16, shuffle=True)
xb, mb = next(iter(train_loader))
print("一個批次：x", tuple(xb.shape), "| 遮罩", tuple(mb.shape), "| 有瑕疵的張數", int((mb.flatten(1).sum(1) > 0).sum()))

## 3. U-Net（三層）
和 D2-1 的自編碼器一樣「往下壓、再往上還原」，差別是**跳接**（skip connection）：
往下每一層的輸出，直接**接（concatenate）**到往上同一層，細節不必擠過瓶頸。
**先別執行**：在投影片「練習：U-Net 每一層的 shape」寫下預測。

In [ ]:
import torch.nn as nn

def block(cin, cout):     # 兩次 3×3 卷積，高寬不變
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.enc1, self.enc2, self.enc3 = block(3, c), block(c, 2 * c), block(2 * c, 4 * c)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), block(2 * c, c)
        self.head = nn.Conv2d(c, 1, 1)         # 每個像素一個 logit

    def forward(self, x, show=False):
        e1 = self.enc1(x)                      # 高寬不變
        e2 = self.enc2(self.pool(e1))          # 高寬 ÷2
        e3 = self.enc3(self.pool(e2))          # 高寬 ÷4
        b = self.mid(self.pool(e3))            # 瓶頸：高寬 ÷8
        # TODO：往上三次：up 放大 → 和同層的 e 接起來（torch.cat，dim=1）→ dec；最後 head
        ...
        if show:
            for name, t in [("x", x), ("e1", e1), ("e2", e2), ("e3", e3), ("b（瓶頸）", b),
                            ("cat(up3(b), e3)", torch.cat([self.up3(b), e3], 1)), ("d3", d3), ("d1", d1), ("out", out)]:
                print(f"{name:18s} {tuple(t.shape)}")
        return out

model = UNet().to(device)
with torch.no_grad():
    model(xb.to(device), show=True)
print("參數量", sum(p.numel() for p in model.parameters()))

## 4. 訓練：BCE ＋ Dice
- **BCE**（逐像素）：每個像素是「瑕疵／背景」的二元分類（DL 課的二元交叉熵）
- **Dice**：瑕疵通常只佔不到 1%，全部猜「背景」BCE 就很低；Dice 只看「瑕疵那幾個像素有沒有對到」

`loss = BCE + (1 − Dice)`

In [ ]:
def dice_coef(prob, m, eps=1.0):
    """prob、m：[B, 1, H, W]；回傳每張的 Dice（平滑版，可微分）。"""
    inter = (prob * m).flatten(1).sum(1)
    return (2 * inter + eps) / (prob.flatten(1).sum(1) + m.flatten(1).sum(1) + eps)

bce = nn.BCEWithLogitsLoss()
def loss_fn(logits, m):
    # TODO：BCE(logits, m) ＋ (1 − 平均 Dice)；Dice 要用 sigmoid(logits)
    ...

CKPT = "unet_pcb1.pt"          # 講師提供的檢查點也用這個檔名；放好就會直接載入、跳過訓練
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    model.load_state_dict(ck["model"]); history = ck.get("history", [])
    print("載入檢查點", CKPT, "（已訓練", len(history), "epochs）")
else:
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train(); total = 0.0
        for xb, mb in train_loader:
            xb, mb = xb.to(device), mb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, mb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(xb)
        history.append(total / len(X_good))
        torch.save({"model": model.state_dict(), "history": history}, CKPT)
        print(f"epoch {epoch:2d}  loss {history[-1]:.4f}  ({time.time()-t0:.0f}s)")
    if device == "cuda":
        print(f"VRAM 峰值 {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

plt.figure(figsize=(4, 2.5)); plt.plot(history); plt.xlabel("epoch"); plt.ylabel("BCE + (1 - Dice)"); plt.title("train loss"); plt.show()

## 5. 在**真實遮罩**上評估
模型只看過合成瑕疵；現在用 VisA 的真實不良品和人工標註的遮罩打分數。
- **IoU** ＝ 重疊 ÷ 聯集；**Dice** ＝ 2 × 重疊 ÷（預測面積＋真實面積）
- 機率 > 0.5 的像素算「瑕疵」

In [ ]:
@torch.no_grad()
def predict(x):
    model.eval()
    return torch.sigmoid(model(x.to(device))).cpu()          # [B, 1, H, W]，0–1

def iou_dice(pred, m):
    """pred、m：0／1 遮罩 [B, 1, H, W] → 每張的 IoU 與 Dice。"""
    # TODO：inter、union；IoU = inter / union；Dice = 2*inter / (pred 面積 + m 面積)；分母為 0 時記 1.0
    ...
    return iou, dice

P = torch.cat([predict(X_bad[i:i + 16]) for i in range(0, len(X_bad), 16)])
pred = (P > 0.5).float()
iou, dice = iou_dice(pred, M_bad)
print(f"真實不良品 {len(X_bad)} 張：平均 IoU {iou.mean():.3f}｜平均 Dice {dice.mean():.3f}")
print(f"IoU 中位數 {iou.median():.3f}；IoU = 0（完全沒對到）的張數 {(iou == 0).sum().item()}")

# 對照：合成瑕疵上的成績（通常好很多，見「風險」頁）
random.seed(123)
xs_, ms_ = zip(*[synth_defect(X_good[i % len(X_good)]) for i in range(64)])
xs_, ms_ = torch.stack(xs_), torch.stack(ms_)
has = ms_.flatten(1).sum(1) > 0
iou_s, dice_s = iou_dice((predict(xs_) > 0.5).float(), ms_)
print(f"合成瑕疵（有瑕疵的 {has.sum().item()} 張）：平均 IoU {iou_s[has].mean():.3f}｜平均 Dice {dice_s[has].mean():.3f}")

In [ ]:
def overlay(x, pred, m=None):
    """紅色＝預測；綠色輪廓＝真實遮罩。"""
    img = x.permute(1, 2, 0).numpy().copy()
    p = pred[0].numpy() > 0
    img[p] = 0.5 * img[p] + 0.5 * np.array([1.0, 0.1, 0.1])
    if m is not None:
        mm = m[0].numpy() > 0
        edge = mm & ~np.pad(mm, 1)[2:, 1:-1] | mm & ~np.pad(mm, 1)[:-2, 1:-1] | mm & ~np.pad(mm, 1)[1:-1, 2:] | mm & ~np.pad(mm, 1)[1:-1, :-2]
        img[edge] = [0.1, 0.9, 0.1]
    return img

order = iou.argsort(descending=True)
pick = [order[0].item(), order[len(order) // 2].item(), order[-1].item()]      # 最好、中間、最差
fig, ax = plt.subplots(1, 3, figsize=(12, 4.2))
for a, i, tag in zip(ax, pick, ["best", "median", "worst"]):
    a.imshow(overlay(X_bad[i], pred[i], M_bad[i])); a.set_title(f"{tag}: IoU {iou[i]:.2f}"); a.axis("off")
plt.tight_layout(); plt.savefig("overlays.png", dpi=120); plt.show()

### 面積：像素 → mm²
維修站要的是 mm²，需要**校正值**：一個像素對應幾 mm。
PCB1 的板子是 HC-SR04 超音波模組，板子約 45 × 20 mm（常見規格，待確認）；
講師在原圖（1404 × 1070）上量到板子約 946 × 418 像素。縮放成 SIZE × SIZE 之後，水平和垂直的比例不一樣，要分開算。

In [ ]:
BOARD_MM, BOARD_PX, ORIG = (45, 20), (946, 418), (1404, 1070)
mm_x = BOARD_MM[0] / BOARD_PX[0] * ORIG[0] / SIZE      # 縮放後，一個像素寬幾 mm
mm_y = BOARD_MM[1] / BOARD_PX[1] * ORIG[1] / SIZE      # 縮放後，一個像素高幾 mm
PX_MM2 = mm_x * mm_y
print(f"縮放後一個像素 ≈ {mm_x:.3f} × {mm_y:.3f} mm ＝ {PX_MM2:.4f} mm²")
# TODO：面積（mm²）＝ 瑕疵像素數 × PX_MM2；分別算預測與真實遮罩
...
for i in pick:
    print(f"第 {i:3d} 張：預測 {area_pred[i]:6.2f} mm²｜真實 {area_true[i]:6.2f} mm²")
print(f"面積平均絕對誤差 {(area_pred - area_true).abs().mean():.2f} mm²")

## 6. Ship：Gradio 維修站工具

In [ ]:
import gradio as gr

def locate(img):
    x = tf(img.convert("RGB"))
    p = (predict(x.unsqueeze(0))[0] > 0.5).float()
    area = p.sum().item() * PX_MM2
    out = (overlay(x, p) * 255).astype(np.uint8)
    return out, f"瑕疵面積約 {area:.2f} mm²（校正：板寬 45 mm）"

demo = gr.Interface(
    fn=locate,
    inputs=gr.Image(type="pil", label="上傳 PCB 影像"),
    outputs=[gr.Image(label="紅色＝預測的瑕疵"), gr.Textbox(label="面積")],
    title="曜嶼電子（虛構）維修站 · 瑕疵位置與面積",
    description="U-Net，只用良品＋合成瑕疵訓練。資料：VisA PCB1（CC BY 4.0）。",
)
demo.launch(share=True)